# Problem Set 5 - Andrew Koren

## Problem 1. LCLS Parameters

Let's use python

<!-- 1. Use formula
2. charge/second * second = charge
3. Each microbunch is one radiation wavelength (thats how coherence works)
4. $L_g = \lambda_u/4\pi\sqrt{3}\rho$ -->

In [113]:
from scipy.constants import e, c, pi

beam_energy_gev = 14.35 #GeV
peak_current = 3500 # A
pulse_len = 230e-15 # s
FEL_param = 5e-4
RMS_angular_div = 1.7e-6 # rad
undulator_wavelen = 0.03 # m
undulator_K = 3.7

beam_energy_eV = beam_energy_gev*1e9
gamma = beam_energy_eV/(511e3)

radiation_wavelength = undulator_wavelen/(2*gamma**2)*(1+undulator_K**2/2)
print(f'1) Radiation Wavelength: {radiation_wavelength:.2e} m')

bunch_charge = peak_current*pulse_len
print(f'2) bunch charge: {bunch_charge:.2e} C')

bunch_len = pulse_len*c
microbunches_per_bunch = bunch_len/radiation_wavelength
bunch_electrons = bunch_charge/e
microbunch_electrons = bunch_electrons/microbunches_per_bunch
print(f'3) electrons per microbunch: {microbunches_per_bunch:.2e}')

gain_len = radiation_wavelength/(4*pi*3**(1/2)*FEL_param)
print(f'4) gain length: {gain_len:.2e} m')

saturation_len = 20 * gain_len
print(f'5) saturation length: {saturation_len:.2e} m')

beam_power = peak_current*beam_energy_eV # watts somehow
radiation_power = beam_power*FEL_param
print(f'6) radiation power: {radiation_power:.2e} w')

rad_freq = 2*pi*c/radiation_wavelength
print(f'7) Frequency Spectrum Width: {rad_freq*FEL_param:.2e} rad/s')

1) Radiation Wavelength: 1.49e-10 m
2) bunch charge: 8.05e-10 C
3) electrons per microbunch: 4.62e+05
4) gain length: 1.37e-08 m
5) saturation length: 2.74e-07 m
6) radiation power: 2.51e+10 w
7) Frequency Spectrum Width: 6.31e+15 rad/s


## Problem 2. Pillbox Cavity

$d = 50 \text{cm}$
$V_0 = 500\text{V}$

a) use the pillbox frequency formula

In [114]:
from scipy.constants import c, pi, epsilon_0, mu_0
from scipy.special import jn_zeros, sinc, j1
import numpy as np
import matplotlib.pyplot as plt

a = 75/200 # m
beta = 0.85
d = 50/100 # m
V0 = 500e3 # V
sigma = 5.9e7

j01 = jn_zeros(0,1)[0]
omega010 = j01*c/a
print(f'a) f = {omega010/(2*pi):.2e} Hz')

v = beta*c
T = sinc(omega010*d/(2*v)/pi) # sinc is normalized by pi
print(f'b) transit time factor: {T:.3f}')

a) f = 3.06e+08 Hz
b) transit time factor: 0.504


c) The transit time factor will increase as $d$ decreases, since $d$ appears in the denominator and will shrink faster than the sine term on top as it decreases (up to a limit)

d) see below

In [115]:
E0 = V0/d
U = 1/2 * epsilon_0*E0**2*d*a**2*(j1(j01))**2
print(f'Stored Energy {U:.2e} J')
skin_depth = (2/(omega010*mu_0*sigma))**(1/2)
print(f'Surface Resistance: {1/(sigma*skin_depth):.2e} Ohms')
Pcycle = (1/2)*pi*epsilon_0*omega010*skin_depth*V0**2*(j1(j01))**2*(a*(a+d))/(d**2)
print(f'Average Surface Power: {Pcycle:.0f} W')
print(f'Q0: {omega010*U/Pcycle:.0f}')
print(f'Shunt Impedance: {V0**2/(2*Pcycle):.2e} Ohms')
print(f'Kinetic energy gain: depends on the charge of the particle being accelerated (qV)')

Stored Energy 8.39e-02 J
Surface Resistance: 4.52e-03 Ohms
Average Surface Power: 8857 W
Q0: 18209
Shunt Impedance: 1.41e+07 Ohms
Kinetic energy gain: depends on the charge of the particle being accelerated (qV)


## Problem 3. Synchrotron Energy Acceptance

a)

$$
% \begin{gathered}
\begin{align*}
        {d^2 \phi \over dn^2} 
   &= - {2\pi h \eta_s \over \gamma_s \beta_s^2} 
        {1 \over mc^2}
        {d \over dn} \Delta W
\\ &= - {2\pi h \eta_s \over \gamma_s \beta_s^2} 
        {q V_{\text{rf}} \over mc^2}
        \left(\sin\phi - \sin\phi_s \right)
\\      {d \phi \over dn}{d^2 \phi \over dn^2} 
   &= - {d \phi \over dn}{2\pi h \eta_s \over \gamma_s \beta_s^2} 
        {q V_{\text{rf}} \over mc^2}
        \left(\sin\phi - \sin\phi_s \right)
\\      \int dn
        {1 \over 2}{d \over dn} \left(d \phi \over dn\right)^2
   &= - \int dn
        {d \phi \over dn}{2\pi h \eta_s \over \gamma_s \beta_s^2} 
        {q V_{\text{rf}} \over mc^2}
        \left(\sin\phi - \sin\phi_s \right)
\\      {1 \over 2}\left(d \phi \over dn\right)^2
   &= - \int d\phi
        {2\pi h \eta_s \over \gamma_s \beta_s^2} 
        {q V_{\text{rf}} \over mc^2}
        \left(\sin\phi - \sin\phi_s \right) + c
\\      {1 \over 2}\left(d \phi \over dn\right)^2
   &-   {2\pi h \eta_s \over \gamma_s \beta_s^2} 
        {q V_{\text{rf}} \over mc^2}
        \left(\cos\phi - \phi\sin\phi_s \right)
    =   c


% \\      \int\left[
%         {1\over 2}{d^2 \phi \over dn^2} 
%     +   {\pi h \eta_s \over \gamma_s \beta_s^2} 
%         {q V_{\text{rf}} \over mc^2}
%         \left(\sin\phi - \sin\phi_s \right) \right] dn
%     =   c
% \\      {1\over 2}{d   \phi \over dn  } 
%     -   {\pi h \eta_s \over \gamma_s \beta_s^2} 
%         {q V_{\text{rf}} \over mc^2}
%         \left(\cos\phi + \phi \sin\phi_s \right)
%     =   c
\end{align*}
$$

b)

$$
\begin{gathered}
{1 \over 2} \left(-{2\pi h \eta_s \over \gamma_s \beta_s^2} {\Delta W \over mc^2} \right)^2 -   {2\pi h \eta_s \over \gamma_s \beta_s^2} 
        {q V_{\text{rf}} \over mc^2}
        \left(\cos\phi - \phi\sin\phi_s \right)
    =   c
\\  \Delta W^2 - {\beta_s^2 \gamma_s mc^2 qV_{\text{rf}} \over \pi h \eta_s}
                 \left(\cos \phi + \phi \sin \phi_s\right) = c
\end{gathered}
$$

c) This is sine convention so $\phi_s=0$ corresponds to no acceleration. As the hint suggests, we'll find the constant value for $\phi = \pi$ and $\Delta W = 0$.

$$
\begin{gathered}
    c = {\beta^2_s \gamma_s mc^2 qV_{\text{rf}} \over \pi h \eta_s}
\\      \Delta W^2 
    -   {\beta_s^2 \gamma_s mc^2 qV_{\text{rf}} \over \pi h \eta_s}
        \cos \phi 
    =   {\beta^2_s \gamma_s mc^2 qV_{\text{rf}} \over \pi h \eta_s}
\\      \Delta W^2 
    -   {\beta_s^2 \gamma_s mc^2 qV_{\text{rf}} \over \pi h \eta_s}
        \left(\cos \phi - 1\right)
    =   0
\\      \Delta W^2 
    -   {2\beta_s^2 \gamma_s mc^2 qV_{\text{rf}} \over \pi h \eta_s}
        \cos^2 \left( \phi \over 2 \right)
    =   0

\end{gathered}
$$



d) We'll compute $\beta_s$ and $\gamma_s$ to get $\Delta W$. Note that the cosine term goes to zero for $\phi = \pm \pi$

$$
    W = (\gamma_s - 1) m_0c^2
\\  \gamma_s = {200 \over 938} + 1 = 1.21
\\  \beta_s = \sqrt{1 - {1 \over \gamma_s^2}} = 0.56
$$

In [122]:
# all energies in MeV
m_p = 938.27
W = 200
gamma = W/m_p + 1
beta = (1-1/gamma**2)**(1/2)
qVrf = 1
h = 80
eta = 0.05
DW2 = 2*beta*beta*gamma*m_p*qVrf/(pi*h*eta)
print(f'Maximum acceptance: {DW2**(1/2)/W:.3f}')

Maximum acceptance: 0.038
